In [1]:
import pandas as pd
import re

# 1. Load data
df = pd.read_csv("Hotel_Reviews.csv")  

df




,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515733,Wurzbachgasse 21 15 Rudolfsheim F nfhaus 1150 ...,168,8/30/2015,8.1,Atlantis Hotel Vienna,Kuwait,no trolly or staff to help you take the lugga...,14,2823,location,2,8,7.0,"[' Leisure trip ', ' Family with older childre...",704 day,48.203745,16.335677
515734,Wurzbachgasse 21 15 Rudolfsheim F nfhaus 1150 ...,168,8/22/2015,8.1,Atlantis Hotel Vienna,Estonia,The hotel looks like 3 but surely not 4,11,2823,Breakfast was ok and we got earlier check in,11,12,5.8,"[' Leisure trip ', ' Family with young childre...",712 day,48.203745,16.335677
515735,Wurzbachgasse 21 15 Rudolfsheim F nfhaus 1150 ...,168,8/19/2015,8.1,Atlantis Hotel Vienna,Egypt,The ac was useless It was a hot week in vienn...,19,2823,No Positive,0,3,2.5,"[' Leisure trip ', ' Family with older childre...",715 day,48.203745,16.335677
515736,Wurzbachgasse 21 15 Rudolfsheim F nfhaus 1150 ...,168,8/17/2015,8.1,Atlantis Hotel Vienna,Mexico,No Negative,0,2823,The rooms are enormous and really comfortable...,25,3,8.8,"[' Leisure trip ', ' Group ', ' Standard Tripl...",717 day,48.203745,16.335677


In [2]:
# 2. Basic text cleaning helpers
def clean_placeholder(text: str) -> str:
    """
    Replace noise like 'No Negative'/'No Positive' with empty string.
    """
    if pd.isna(text):
        return ""
    text = text.strip()
    if text in ["No Negative", "No Positive"]:
        return ""
    return text

def clean_tags(raw_tags: str) -> list:
    """
    Convert the tags string to a Python list and drop useless phrases
    like 'Submitted from a mobile device'.
    """
    if pd.isna(raw_tags):
        return []

    # Many versions of this dataset store tags like "[' Leisure trip ', ' Couple ', ...]"
    # We'll extract the inner words between quotes.
    # Simple regex approach:
    tags = re.findall(r"'([^']+)'", str(raw_tags))
    tags = [t.strip() for t in tags]

    # Remove noise tags
    noise_tags = {"Submitted from a mobile device"}
    tags = [t for t in tags if t not in noise_tags]

    return tags

def shorten_address(addr: str) -> str:
    """
    Optional: Shorten full address to something like 'Amsterdam, Netherlands'.
    If you don't need city/country in text, you can even return an empty string.
    """
    if pd.isna(addr):
        return ""
    addr = addr.strip()
    # Heuristic: city and country are usually at the end of the address
    parts = addr.split(" ")
    # Last two tokens often are 'City' and 'Country' but structure varies.
    # For now, just return the full address; you can customize later.
    return addr

In [3]:
# 3. Apply cleaning to text columns
df["Negative_Review_clean"] = df["Negative_Review"].apply(clean_placeholder)
df["Positive_Review_clean"] = df["Positive_Review"].apply(clean_placeholder)
df["Tags_list"] = df["Tags"].apply(clean_tags)

In [4]:
# 4. Build document_text for each row
def build_document_text(row):
    hotel = str(row.get("Hotel_Name", "")).strip()
    address = shorten_address(row.get("Hotel_Address", ""))
    nationality = str(row.get("Reviewer_Nationality", "")).strip()
    review_date = str(row.get("Review_Date", "")).strip()
    avg_score = row.get("Average_Score", None)
    reviewer_score = row.get("Reviewer_Score", None)
    pos = row.get("Positive_Review_clean", "")
    neg = row.get("Negative_Review_clean", "")
    tags = row.get("Tags_list", [])

    parts = []

    if hotel:
        parts.append(f"Hotel: {hotel}.")
    if address:
        parts.append(f"Address: {address}.")
    if review_date:
        parts.append(f"Review date: {review_date}.")
    if nationality:
        parts.append(f"Reviewer nationality: {nationality}.")
    if avg_score is not None:
        parts.append(f"Average hotel score: {avg_score}.")
    if reviewer_score is not None:
        parts.append(f"Reviewer score: {reviewer_score}.")

    if tags:
        tag_str = ", ".join(tags)
        parts.append(f"Tags: {tag_str}.")

    # Add positive/negative text only if not empty
    if pos:
        parts.append(f"Positive review: {pos}")
    if neg:
        parts.append(f"Negative review: {neg}")

    # Join all parts into one document string
    return " ".join(parts)

df["document_text"] = df.apply(build_document_text, axis=1)


In [5]:
# 5. Build metadata dict per row (for RAG filters)
def build_metadata(row):
    return {
        "hotel_name": str(row.get("Hotel_Name", "")).strip(),
        "hotel_address": str(row.get("Hotel_Address", "")).strip(),
        "review_date": str(row.get("Review_Date", "")).strip(),
        "reviewer_nationality": str(row.get("Reviewer_Nationality", "")).strip(),
        "average_score": float(row.get("Average_Score", 0.0)),
        "reviewer_score": float(row.get("Reviewer_Score", 0.0)),
        "tags": row.get("Tags_list", []),
        "lat": float(row.get("lat", 0.0)),
        "lng": float(row.get("lng", 0.0)),
    }

df["metadata"] = df.apply(build_metadata, axis=1)

In [7]:
# 6. (Optional) keep only the needed columns for the RAG index
rag_df = df[["document_text", "metadata"]].copy()

# Optionally inspect a few examples
print(rag_df.iloc[0]["document_text"])
print(rag_df.iloc[0]["metadata"])



Hotel: Hotel Arena. Address: s Gravesandestraat 55 Oost 1092 AA Amsterdam Netherlands. Review date: 8/3/2017. Reviewer nationality: Russia. Average hotel score: 7.7. Reviewer score: 2.9. Tags: Leisure trip, Couple, Duplex Double Room, Stayed 6 nights. Positive review: Only the park outside of the hotel was beautiful Negative review: I am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place I made my booking via booking com We stayed for 6 nights in this hotel from 11 to 17 July Upon arrival we were placed in a small room on the 2nd floor of the hotel It turned out that this was not the room we booked I had specially reserved the 2 level duplex room so that we would have a big windows and high ceilings The room itself was ok if you don t mind the broken window that can not be closed hello rain and a mini fridge that contained some sort of a bio weapon at least i guessed so by the smell of it I i

In [8]:
rag_df

,document_text,metadata
0,Hotel: Hotel Arena. Address: s Gravesandestraa...,"{'hotel_name': 'Hotel Arena', 'hotel_address':..."
1,Hotel: Hotel Arena. Address: s Gravesandestraa...,"{'hotel_name': 'Hotel Arena', 'hotel_address':..."
2,Hotel: Hotel Arena. Address: s Gravesandestraa...,"{'hotel_name': 'Hotel Arena', 'hotel_address':..."
3,Hotel: Hotel Arena. Address: s Gravesandestraa...,"{'hotel_name': 'Hotel Arena', 'hotel_address':..."
4,Hotel: Hotel Arena. Address: s Gravesandestraa...,"{'hotel_name': 'Hotel Arena', 'hotel_address':..."
...,...,...
515733,Hotel: Atlantis Hotel Vienna. Address: Wurzbac...,"{'hotel_name': 'Atlantis Hotel Vienna', 'hotel..."
515734,Hotel: Atlantis Hotel Vienna. Address: Wurzbac...,"{'hotel_name': 'Atlantis Hotel Vienna', 'hotel..."
515735,Hotel: Atlantis Hotel Vienna. Address: Wurzbac...,"{'hotel_name': 'Atlantis Hotel Vienna', 'hotel..."
515736,Hotel: Atlantis Hotel Vienna. Address: Wurzbac...,"{'hotel_name': 'Atlantis Hotel Vienna', 'hotel..."


In [9]:
! pip install sentence-transformers

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model once at startup
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert document texts to a list
documents = rag_df["document_text"].tolist()

# Compute embeddings (batched internally)
embeddings = embed_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # helps cosine similarity
)

print(embeddings.shape)  # (num_documents, embedding_dim)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/8059 [00:00<?, ?it/s]

(515738, 384)


In [9]:
rag_df = rag_df.reset_index(drop=True)
rag_df["id"] = rag_df.index.astype(str)  # simple string ids

# Store embeddings in a NumPy array
embedding_matrix = embeddings  # shape: (N, D)

# Optional: build a list of dicts ready for insertion into a vector DB
records = []
for i, row in rag_df.iterrows():
    rec = {
        "id": row["id"],
        "embedding": embedding_matrix[i],
        "document_text": row["document_text"],
        "metadata": row["metadata"],
    }
    records.append(rec)

# Quick sanity check
print(records[0]["id"])
print(records[0]["embedding"][:5])   # first 5 dimensions
print(records[0]["document_text"])
print(records[0]["metadata"])

NameError: name 'embeddings' is not defined

In [10]:
# Save embeddings
np.save("hotel_embeddings.npy", embedding_matrix)

# Save the dataframe (ids, document_text, metadata)
rag_df.to_parquet("rag_df.parquet")

NameError: name 'np' is not defined

In [19]:
! pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ------------- -------------------------- 6.3/18.9 MB 42.8 MB/s eta 0:00:01
   ----------------------------- ---------- 14.2/18.9 MB 41.0 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 38.4 MB/s eta 0:00:00


In [11]:
"""import chromadb
from chromadb.utils import embedding_functions
import numpy as np

# Create an in-memory Chroma client (for local experiments)
client = chromadb.Client()

# Create or get a collection for hotel reviews
collection = client.create_collection(name="hotel_reviews")
"""

import faiss
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Load your data (assuming you saved after Step 2)
# ------------------------------------------------------------
rag_df = pd.read_parquet("rag_df.parquet")
embedding_matrix = np.load("hotel_embeddings.npy")

print(f"Loaded {len(rag_df)} documents")
print(f"Embedding matrix shape: {embedding_matrix.shape}")

# ------------------------------------------------------------
# Build FAISS index
# ------------------------------------------------------------
dimension = embedding_matrix.shape[1]  # e.g., 384 for all-MiniLM-L6-v2

Loaded 515738 documents
Embedding matrix shape: (515738, 384)


This collection is your vector index: it will hold embeddings, texts, and metadata, and support similarity search.

Add your documents + embeddings
Chroma expects:

ids: list of strings

embeddings: list/array of vectors

documents: list of original texts

metadatas: list of dicts

In [12]:
"""i_d = rag_df["id"].tolist()
documents = rag_df["document_text"].tolist()
metadatas = rag_df["metadata"].tolist()  # each row is already a dict
embeddings_list = embedding_matrix.tolist()  # convert NumPy array to list of lists

# Add everything to the collection (you can also do this in chunks if it's huge)
collection.add(
    ids=i_d,
    embeddings=embeddings_list,
    documents=documents,
    metadatas=metadatas,
)
"""

# Create a flat (exact search) index with L2 distance
# For cosine similarity, use IndexFlatIP and normalize embeddings first
index = faiss.IndexFlatL2(dimension)

# Add all embeddings at once (no batch size limit)
index.add(embedding_matrix)

print(f"FAISS index now contains {index.ntotal} vectors")

# ------------------------------------------------------------
# Save FAISS index to disk
# ------------------------------------------------------------
faiss.write_index(index, "hotel_reviews.faiss")
print("FAISS index saved to 'hotel_reviews.faiss'")

# ------------------------------------------------------------
# Save the dataframe (for metadata and text retrieval)
# ------------------------------------------------------------
rag_df.to_parquet("rag_df_faiss.parquet")
print("Dataframe saved to 'rag_df_faiss.parquet'")

FAISS index now contains 515738 vectors
FAISS index saved to 'hotel_reviews.faiss'
Dataframe saved to 'rag_df_faiss.parquet'


How to query FAISS.
When you want to retrieve reviews for a question:

In [13]:
"""Test a simple semantic query
Let’s verify retrieval works before wiring in the LLM."""
"""
query_text = "What do guests complain about at Hotel Arena?"

results = collection.query(
    query_texts=[query_text],
    n_results=5,  # top K
)

# Inspect what came back
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("Hotel:", meta.get("hotel_name"))
    print("Score:", meta.get("reviewer_score"))
    print("Review snippet:", doc[:300], "...")
    print("-" * 80)
"""

""" Now let's do the same retrieval test using our FAISS index and dataframe directly, without ChromaDB.
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Load FAISS index and dataframe
# ------------------------------------------------------------
index = faiss.read_index("hotel_reviews.faiss")
rag_df = pd.read_parquet("rag_df_faiss.parquet")

# Load embedding model (same one used for indexing)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ------------------------------------------------------------
# User question
# ------------------------------------------------------------
user_question = "What do guests complain about at Hotel Arena?"

# Embed the question
query_embedding = embed_model.encode([user_question], convert_to_numpy=True)

# ------------------------------------------------------------
# Search FAISS
# ------------------------------------------------------------
k = 5  # top 5 results
distances, indices = index.search(query_embedding, k)

print(f"Retrieved {k} reviews:")
print("Indices:", indices[0])
print("Distances:", distances[0])

# ------------------------------------------------------------
# Retrieve documents and metadata using the indices
# ------------------------------------------------------------
retrieved_docs = []
retrieved_metas = []

for idx in indices[0]:
    doc = rag_df.iloc[idx]["document_text"]
    meta = rag_df.iloc[idx]["metadata"]
    
    retrieved_docs.append(doc)
    retrieved_metas.append(meta)
    
    print("-" * 80)
    print("Hotel:", meta.get("hotel_name"))
    print("Score:", meta.get("reviewer_score"))
    print("Review snippet:", doc[:300], "...")
"""

' Now let\'s do the same retrieval test using our FAISS index and dataframe directly, without ChromaDB.\nimport faiss\nimport numpy as np\nimport pandas as pd\nfrom sentence_transformers import SentenceTransformer\n\n# ------------------------------------------------------------\n# Load FAISS index and dataframe\n# ------------------------------------------------------------\nindex = faiss.read_index("hotel_reviews.faiss")\nrag_df = pd.read_parquet("rag_df_faiss.parquet")\n\n# Load embedding model (same one used for indexing)\nembed_model = SentenceTransformer("all-MiniLM-L6-v2")\n\n# ------------------------------------------------------------\n# User question\n# ------------------------------------------------------------\nuser_question = "What do guests complain about at Hotel Arena?"\n\n# Embed the question\nquery_embedding = embed_model.encode([user_question], convert_to_numpy=True)\n\n# ------------------------------------------------------------\n# Search FAISS\n# --------------

In [30]:
! pip install groq

Step 4: Query → Retrieve → Prompt → Generate (using FAISS)

In [14]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from groq import Groq

# ------------------------------------------------------------
# Setup: Load FAISS index, dataframe, and embedding model
# ------------------------------------------------------------
index = faiss.read_index("hotel_reviews.faiss")
rag_df = pd.read_parquet("rag_df_faiss.parquet")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"FAISS index contains {index.ntotal} vectors")
print(f"Dataframe has {len(rag_df)} rows")

c:\Users\tariq\OneDrive\Documents\Python\Hotel Reviews\Hotel Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6045.95it/s]


FAISS index contains 515738 vectors
Dataframe has 515738 rows


In [15]:
# ------------------------------------------------------------
# 4.1 User asks a question
# ------------------------------------------------------------
user_question = "What do guests complain about at Hotel Arena?"

# ------------------------------------------------------------
# 4.2 Retrieve relevant reviews from FAISS
# ------------------------------------------------------------
# Embed the question
query_embedding = embed_model.encode([user_question], convert_to_numpy=True)

# Search FAISS for top K similar reviews
k = 3
distances, indices = index.search(query_embedding, k)

print(f"\nRetrieved {k} reviews with indices: {indices[0]}")

# Retrieve documents and metadata from the dataframe
retrieved_docs = []
retrieved_metas = []

for idx in indices[0]:
    doc = rag_df.iloc[idx]["document_text"]
    meta = rag_df.iloc[idx]["metadata"]
    retrieved_docs.append(doc)
    retrieved_metas.append(meta)


Retrieved 3 reviews with indices: [   229 294451 294447]


In [16]:
# ------------------------------------------------------------
# 4.3 Build the prompt for the LLM
# ------------------------------------------------------------
system_message = """You are a helpful hotel review assistant. 
Answer the user's question using ONLY the reviews provided in the context below.
Summarize patterns, mention frequent issues or praises, and quote short phrases from reviews.
If the context does not contain enough information to answer, say so clearly.
Always be factual and grounded in the provided reviews."""

# Build context from retrieved reviews
context_parts = []
for i, (doc, meta) in enumerate(zip(retrieved_docs, retrieved_metas)):
    hotel_name = meta.get("hotel_name", "Unknown")
    reviewer_score = meta.get("reviewer_score", "N/A")
    review_date = meta.get("review_date", "N/A")
    
    # Truncate doc to ~500 chars to keep prompt manageable
    doc_snippet = doc[:500] + "..." if len(doc) > 500 else doc
    
    context_parts.append(
        f"[Review {i+1}]\n"
        f"Hotel: {hotel_name}\n"
        f"Score: {reviewer_score}\n"
        f"Date: {review_date}\n"
        f"Text: {doc_snippet}\n"
    )

context = "\n".join(context_parts)

# Full prompt
prompt = f"""CONTEXT (Retrieved Reviews):
{context}

QUESTION: {user_question}

ANSWER:"""


In [17]:
# ------------------------------------------------------------
# 4.4 Call the LLM to generate the answer
# ------------------------------------------------------------
# Using Groq API

# Initialize the client
client = Groq(api_key="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")  # replace with your actual key

# Call the API
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # or llama-3.1-8b-instant for faster/cheaper
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
    max_tokens=300,
)

# Extract the answer
answer = response.choices[0].message.content

# ------------------------------------------------------------
# 4.5 Display the answer with citations
# ------------------------------------------------------------
print("=" * 80)
print("QUESTION:", user_question)
print("=" * 80)
print("\nANSWER:")
print(answer)
print("\n" + "=" * 80)
print("SOURCES (Retrieved Reviews):")
for i, meta in enumerate(retrieved_metas):
    print(f"  [{i+1}] {meta.get('hotel_name')}, "
          f"Score: {meta.get('reviewer_score')}, "
          f"Date: {meta.get('review_date')}")
print("=" * 80)

QUESTION: What do guests complain about at Hotel Arena?

ANSWER:
There is not enough information to answer the question about guest complaints at Hotel Arena, as only one review is provided for Hotel Arena and it is a positive review with no complaints mentioned. The reviewer from Argentina states "We loved our stay at hotel arena" and praises the staff, but does not mention any negative aspects of their stay.

SOURCES (Retrieved Reviews):
  [1] Hotel Arena, Score: 10.0, Date: 9/7/2015
  [2] Arenas Atiram Hotels, Score: 9.2, Date: 7/23/2016
  [3] Arenas Atiram Hotels, Score: 2.9, Date: 8/25/2016
